# 02 - 容器模块：Sequential, ModuleList, ModuleDict

## 学习目标

1. 掌握 `nn.Sequential` 的两种构建方式
2. 理解 `nn.ModuleList` 与 Python list 的区别
3. 学会使用 `nn.ModuleDict` 进行动态模块选择
4. 了解 `ParameterList` 和 `ParameterDict`
5. 用 AlexNet 风格的结构综合演示

---

In [ ]:
import torch
import torch.nn as nn
from collections import OrderedDict

## 1. nn.Sequential

`nn.Sequential` 是一个**有序容器**，数据会按照定义的顺序依次通过每个模块。

适用场景：前向传播是**简单的线性流水线**（一层接一层）。

### 方式一：直接传入模块

In [ ]:
# 方式一：直接传入子模块（自动编号 0, 1, 2, ...）
seq_model_v1 = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
)

print("方式一（自动编号）:")
print(seq_model_v1)

# 测试前向传播
x = torch.randn(1, 1, 28, 28)
out = seq_model_v1(x)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {out.shape}")

### 方式二：使用 OrderedDict（推荐）

可以给每一层起一个有意义的名字，方便后续访问和调试。

In [ ]:
# 方式二：使用 OrderedDict（带命名）
seq_model_v2 = nn.Sequential(OrderedDict([
    ('conv1', nn.Conv2d(1, 16, kernel_size=3, padding=1)),
    ('relu1', nn.ReLU()),
    ('pool1', nn.MaxPool2d(2)),
]))

print("方式二（OrderedDict）:")
print(seq_model_v2)

# 通过名称访问子模块
print(f"\n通过名称访问: {seq_model_v2.conv1}")

# 通过索引访问子模块
print(f"通过索引访问: {seq_model_v2[0]}")

**关键观察：**

1. 方式一的层名是 `0, 1, 2, ...`，方式二可以自定义名字
2. `Sequential` 的 `forward` 方法已经写好了：依次调用每个子模块
3. 使用 `OrderedDict` 命名后，可以用 `model.conv1` 来访问指定层

## 2. nn.ModuleList

`nn.ModuleList` 是一个**列表容器**，用来存储子模块。

**关键区别：** 它不像 `Sequential` 那样自动定义前向传播，你需要自己在 `forward` 中遍历。

### ModuleList vs Python list

In [ ]:
# 错误示范：使用 Python list
class BadModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 用普通 list 存储层 -> 参数不会被注册！
        self.layers = [nn.Linear(10, 10) for _ in range(3)]

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


bad_model = BadModel()
print("Python list 方式:")
print(f"参数数量: {sum(p.numel() for p in bad_model.parameters())}")
print(f"_modules: {dict(bad_model._modules)}")
print("参数为 0！优化器无法更新这些层！")

In [ ]:
# 正确做法：使用 nn.ModuleList
class GoodModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 用 ModuleList 存储层 -> 参数会被正确注册
        self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


good_model = GoodModel()
print("ModuleList 方式:")
print(f"参数数量: {sum(p.numel() for p in good_model.parameters())}")
print(f"\n_modules 中的子模块:")
for name, mod in good_model._modules.items():
    print(f"  {name}: {mod}")

**重要结论：**

- **Python list**：层不会被注册，`parameters()` 无法遍历到，优化器无法更新
- **nn.ModuleList**：层会被正确注册到 `_modules` 中

### ModuleList 的常用操作

In [ ]:
# ModuleList 支持列表式操作
module_list = nn.ModuleList()

# append: 添加模块
module_list.append(nn.Linear(10, 20))
module_list.append(nn.ReLU())

# extend: 批量添加
module_list.extend([nn.Linear(20, 5), nn.Sigmoid()])

# insert: 指定位置插入
module_list.insert(1, nn.BatchNorm1d(20))

print(f"ModuleList 长度: {len(module_list)}")
for i, mod in enumerate(module_list):
    print(f"  [{i}]: {mod}")

# 支持索引访问
print(f"\n第 0 个模块: {module_list[0]}")

## 3. nn.ModuleDict

`nn.ModuleDict` 是一个**字典容器**，通过名称存取子模块。

适用场景：需要根据条件**动态选择**不同的模块。

In [ ]:
class DynamicModel(nn.Module):
    """根据配置动态选择激活函数的模型。"""

    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 10)
        # 用 ModuleDict 存储多种激活函数
        self.activations = nn.ModuleDict({
            'relu': nn.ReLU(),
            'leaky_relu': nn.LeakyReLU(0.1),
            'gelu': nn.GELU(),
            'sigmoid': nn.Sigmoid(),
        })

    def forward(self, x, act_name='relu'):
        x = self.linear(x)
        # 动态选择激活函数
        x = self.activations[act_name](x)
        return x


model = DynamicModel()
x = torch.randn(2, 10)

print("使用不同激活函数:")
for act in ['relu', 'leaky_relu', 'gelu', 'sigmoid']:
    out = model(x, act_name=act)
    print(f"  {act:12s} -> 输出范围: [{out.min().item():.3f}, {out.max().item():.3f}]")

print(f"\n总参数量: {sum(p.numel() for p in model.parameters())}")

## 4. ParameterList 和 ParameterDict

类似于 `ModuleList` 和 `ModuleDict`，但用来存储 `nn.Parameter` 而非子模块。

In [ ]:
class ParamDemo(nn.Module):
    def __init__(self):
        super().__init__()
        # ParameterList: 存储一组参数
        self.weights = nn.ParameterList(
            [nn.Parameter(torch.randn(3, 3)) for _ in range(3)]
        )
        # ParameterDict: 按名称存储参数
        self.biases = nn.ParameterDict({
            'layer1': nn.Parameter(torch.zeros(3)),
            'layer2': nn.Parameter(torch.zeros(3)),
        })

    def forward(self, x):
        return x


param_demo = ParamDemo()

print("所有参数:")
for name, param in param_demo.named_parameters():
    print(f"  {name}: shape={param.shape}")

**适用场景：**

- 当你需要在模型中存储一组**自定义参数**（不属于任何标准层）时
- 例如：多头注意力中的多组投影矩阵

## 5. AlexNet 风格的结构示例

经典的 AlexNet 使用 `features`（特征提取）+ `classifier`（分类器）的结构。这是实际工程中非常常见的模式。

In [ ]:
class MiniAlexNet(nn.Module):
    """简化版 AlexNet 结构：features + classifier。"""

    def __init__(self, num_classes=10):
        super().__init__()
        # 特征提取部分（使用 Sequential）
        self.features = nn.Sequential(OrderedDict([
            ('conv1', nn.Conv2d(1, 32, kernel_size=3, padding=1)),
            ('relu1', nn.ReLU()),
            ('pool1', nn.MaxPool2d(2)),
            ('conv2', nn.Conv2d(32, 64, kernel_size=3, padding=1)),
            ('relu2', nn.ReLU()),
            ('pool2', nn.MaxPool2d(2)),
        ]))
        # 分类器部分（使用 Sequential）
        self.classifier = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(64 * 7 * 7, 256)),
            ('relu3', nn.ReLU()),
            ('dropout', nn.Dropout(0.5)),
            ('fc2', nn.Linear(256, num_classes)),
        ]))

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # 展平
        x = self.classifier(x)
        return x


model = MiniAlexNet(num_classes=10)
print(model)

In [ ]:
# 测试前向传播
x = torch.randn(4, 1, 28, 28)
output = model(x)
print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")

# 分别统计 features 和 classifier 的参数量
feat_params = sum(p.numel() for p in model.features.parameters())
cls_params = sum(p.numel() for p in model.classifier.parameters())
print(f"\nfeatures 参数量: {feat_params:,}")
print(f"classifier 参数量: {cls_params:,}")
print(f"总参数量: {feat_params + cls_params:,}")

**关键观察：**

1. `features` 和 `classifier` 各自是一个 `Sequential`，模型结构层次清晰
2. 可以通过 `model.features.conv1` 访问具体某一层
3. 可以单独统计每个部分的参数量
4. 迁移学习时，通常只替换 `classifier` 部分

## 6. 容器对比总结

| 容器 | 特点 | 何时使用 |
|------|------|----------|
| `Sequential` | 有序，自动前向传播 | 简单的线性流水线 |
| `ModuleList` | 列表式，需手写 forward | 需要循环/条件控制的层 |
| `ModuleDict` | 字典式，按名称访问 | 需要动态选择模块 |
| `ParameterList` | 列表式参数容器 | 存储一组自定义参数 |
| `ParameterDict` | 字典式参数容器 | 按名称管理自定义参数 |

## 7. 练习题

### 练习 1：使用 Sequential 搭建一个 3 层 MLP

输入 784 → 隐藏 256 → 隐藏 128 → 输出 10，每层之间加 ReLU。

In [ ]:
# TODO: 使用 nn.Sequential + OrderedDict 创建 MLP
# mlp = nn.Sequential(OrderedDict([...]))

# 测试代码
# x = torch.randn(2, 784)
# output = mlp(x)
# print(f"输出形状: {output.shape}")  # 应为 [2, 10]

### 练习 2：使用 ModuleList 构建可配置深度的网络

创建一个模型，根据传入的 `num_layers` 参数动态构建多层 Linear + ReLU。

In [ ]:
class FlexibleMLP(nn.Module):
    """可配置深度的 MLP。"""

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        # TODO: 使用 ModuleList 构建 num_layers 层网络
        pass

    def forward(self, x):
        # TODO: 遍历 ModuleList 进行前向传播
        pass


# 测试代码
# model = FlexibleMLP(784, 128, 10, num_layers=4)
# x = torch.randn(2, 784)
# output = model(x)
# print(f"输出形状: {output.shape}")
# print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

### 练习 3：使用 ModuleDict 实现可选归一化层

创建一个模型，通过参数选择使用 BatchNorm、LayerNorm 或不使用归一化。

In [ ]:
class NormSelectModel(nn.Module):
    """可选归一化的模型。"""

    def __init__(self, hidden_dim):
        super().__init__()
        # TODO: 使用 ModuleDict 存储不同的归一化层
        pass

    def forward(self, x, norm_type='batch'):
        # TODO: 根据 norm_type 选择归一化方式
        pass


# 测试代码
# model = NormSelectModel(hidden_dim=64)
# x = torch.randn(4, 64)
# for norm in ['batch', 'layer']:
#     out = model(x, norm_type=norm)
#     print(f"{norm}: shape={out.shape}")

## 8. 小结

### 核心要点

1. **`nn.Sequential`**：最简单的容器，适合线性流水线，支持 OrderedDict 命名
2. **`nn.ModuleList`**：列表式容器，必须自己写 `forward`，适合动态深度网络
3. **`nn.ModuleDict`**：字典式容器，适合动态模块选择
4. **永远不要用 Python list/dict 存储子模块**——参数不会被注册

### 设计建议

- 固定结构 → `Sequential`
- 重复结构（如 N 层 Transformer block） → `ModuleList`
- 条件/动态选择 → `ModuleDict`
- AlexNet 风格（`features` + `classifier`） → 嵌套 `Sequential`

### 下一步

在下一个 notebook 中，我们将学习 PyTorch 中**常用的网络层**（卷积、池化、归一化、激活函数等）。